## 🔍 Preprocessing and Training Data Development - Wildfire Preprocessing and Training

In this notebook, we begin setting up the framework for the modeling step. The primary is to: 

- One-hot-encode categorical features such as Region and Season
-  Scale numerical features like Temperature, Humidity, WindSpeed, and Precipitation using StandardScaler
- Split the dataset into training and testing subsets using train_test_split
- Apply cross-validation to assess the score but this is situational



In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [2]:
file_path = "..\\data\\wildfire_data_eda.csv" 
wildfire = pd.read_csv(file_path)

In [3]:
wildfire.sample(5)

,Latitude,Longitude,ArchiveYear,Started,AcresBurned,Counties,WildfireOccurred,Region,Season,Date,Temperature,Humidity,WindSpeed,Precipitation
380,35.601393,-117.667576,2014,2014-02-18 05:28:00+00:00,0,Kern,0,Southern California,Winter,2/18/2014,13.8,39.0,7.4,0.0
1224,37.065310,-121.685770,2016,2016-08-12 16:00:00+00:00,25,Santa Clara,1,Western California,Summer,8/12/2016,19.1,68.0,6.2,0.0
2806,34.098191,-118.478717,2019,2019-10-28 06:46:55+00:00,745,Los Angeles,1,Southern California,Fall,10/28/2019,17.6,25.0,12.4,0.3
696,36.615762,-121.470971,2015,2015-02-02 16:27:00+00:00,0,Monterey,0,Western California,Winter,2/2/2015,12.1,72.0,3.3,0.0
1042,37.252959,-119.883475,2016,2016-03-07 21:56:00+00:00,0,Madera,0,Central California,Winter,3/7/2016,8.5,80.0,13.8,22.2


## One-hot encode / Dummy feature

#### In the first step we only need to One-hot-encode the Region and Season columns, as they are categorical values, counties would add too much complexity and add too many features, they are already mapped to a broader Region. 

In [4]:
# One-hot encode 'Region' and 'Season' columns
categorical_cols = ['Region', 'Season']

# Use pandas get_dummies for simplicity, drop the first category to avoid dummy variable trap / multicollinearity
wildfire_en= pd.get_dummies(wildfire, columns=categorical_cols, drop_first=True, dtype=int)

# Display the first few rows to verify
wildfire_en.head()

,Latitude,Longitude,ArchiveYear,Started,AcresBurned,Counties,WildfireOccurred,Date,Temperature,Humidity,WindSpeed,Precipitation,Region_Northern California,Region_Southern California,Region_Western California,Season_Spring,Season_Summer,Season_Winter
0,40.785933,-121.017469,2013,2013-01-01 08:14:00+00:00,0,Lassen,0,1/1/2013,-11.0,79.0,7.6,0.0,1,0,0,0,0,1
1,33.469231,-115.114638,2013,2013-01-02 16:51:00+00:00,0,Riverside,0,1/2/2013,7.2,23.0,15.0,0.0,0,1,0,0,0,1
2,40.816333,-121.838142,2013,2013-01-04 15:07:00+00:00,0,Shasta,0,1/4/2013,-2.2,76.0,7.9,0.0,1,0,0,0,0,1
3,34.186179,-118.661168,2013,2013-01-06 17:30:00+00:00,0,Los Angeles,0,1/6/2013,9.2,76.0,6.2,0.9,0,1,0,0,0,1
4,37.507826,-122.105244,2013,2013-01-08 05:47:00+00:00,0,Alameda,0,1/8/2013,9.1,84.0,3.3,0.0,0,0,1,0,0,1


In [5]:
print(f"Temperature Range: ({wildfire['Temperature'].min()}, {wildfire['Temperature'].max()})")
print(f"Humidity Range: ({wildfire['Humidity'].min()}, {wildfire['Humidity'].max()})")
print(f"Wind Speed Range: ({wildfire['WindSpeed'].min()}, {wildfire['WindSpeed'].max()})")
print(f"Precipitation Range: ({wildfire['Precipitation'].min()}, {wildfire['Precipitation'].max()})")

Temperature Range: (-20.2, 40.7)
Humidity Range: (5.0, 100.0)
Wind Speed Range: (1.9, 41.7)
Precipitation Range: (0.0, 74.0)


## Scaling features

#### Now we need to do the next step, which is Standardization for scaling, weather features like 
- `Temperature` (-20.2, 40.7)
- `Humidity` (5.0, 100.0), 
- `WindSpeed` (1.9, 41.7), 
- `Precipitation` (0.0, 74.0),

They are not on the same scale, during modeling without scaling, the model might give more weight to Humidity due to a larger max value rather than importance, by applying StandardScaler we get all them on equal weight numerically.

In [6]:
# Select the columns to scale
scale_cols = ['Temperature', 'Humidity', 'WindSpeed', 'Precipitation']

scaler = StandardScaler()

# Fit and transform the scaler on the selected columns
scaled_data = scaler.fit_transform(wildfire_en[scale_cols])

# Replace the original columns with the scaled data
wildfire_en[scale_cols] = pd.DataFrame(scaled_data, columns=scale_cols, index=wildfire_en.index)
wildfire_en[scale_cols] = wildfire_en[scale_cols].round(2)

In [7]:
wildfire_en[scale_cols].sample(5)

,Temperature,Humidity,WindSpeed,Precipitation
975,-1.65,0.94,-0.80,-0.22
1854,-0.25,-0.05,-0.51,-0.22
1565,0.33,-0.53,1.07,-0.22
2517,-0.20,-0.48,-0.70,-0.22
1408,-1.16,1.08,1.21,0.23


## Test/Train Split

The final step in this notebook is to split the data into training and testing subsets. This ensures that model evaluation is performed on data that was not seen during training, for an unbiassed assement of the model performance.

We will use a 70/30 split, where 70% of the data is used for training and 30% for testing. 

**Steps:**
- Define the feature matrix `X` and target vector `y`
- Split the data into training and testing sets using `train_test_split`
- The scaler should be fit only on the training data and then applied to the test data

This prepares the data for the modeling phase

In [8]:
round(len(wildfire) * 0.7), round(len(wildfire) * 0.3)

(2005, 859)

#### For predicting WildfireOccurred, use features that are available before a wildfire occurs that is relevant for prediction. 

Scaled weather features: 
- `Temperature`
- `Humidity`
- `WindSpeed`
- `Precipitation`

#### One-hot encoded region and season columns

Don't need WildfireOccured (target variable)
- `Started`,`Date` future info
- `Counties`, `ArchiveYear` we have region info, year can be used for yearly trends but year doesn't necessarily determine much. 
- `AcresBurned` which is an outcome not a predictor.



In [9]:
# Define feature matrix X and target vector y
X = wildfire_en.drop(columns=['WildfireOccurred', 'Started', 'Date', 'Counties', 'ArchiveYear'])
y = wildfire_en['WildfireOccurred']

# Split the data into training and testing sets (70/30 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Display the shape of the resulting splits
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (2004, 13)
X_test shape: (860, 13)
y_train shape: (2004,)
y_test shape: (860,)


In [11]:
wildfire_en.to_csv('..\\data\\wildfire_data_preprocessed.csv', index = False)

# Summary

In this Preprocessing and Training Data Development notebook, I prepared the wildfire dataset for modeling by applying data transformation techniques. First, I one-hot encoded the categorical features `Region` and `Season` to numerical format for machine learning. Next, I standardized the numerical weather features (`Temperature`, `Humidity`, `WindSpeed`, and `Precipitation`) using `StandardScaler` to ensure all features were on the same scale, thereby preventing any features from dominating the model due to their magnitude.

After preprocessing, I defined the feature matrix `X` and the target vector `y`, carefully excluding columns that would not be available or relevant for prediction, such as `Started`, `Date`, `Counties`, `ArchiveYear`, and `AcresBurned`. Finally, I split the dataset into training and testing subsets using a 70/30 split, ensuring that the model can be evaluated on unseen data in the next phase. I have essentially completed the necessary steps required for this notebook. Although it was a short notebook, the remaining part will be much longer. Modeling comes in the next notebook, which will be used for selecting the best model and assessing the best performance accuracy for the chosen supervised machine learning model. 